In [1]:
import pandas as pd          # main data tool (like Excel in Python)
import numpy as np           # math & number operations
import matplotlib.pyplot as plt  # basic charts
import seaborn as sns        # nicer looking charts
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # show ALL columns
print('All libraries imported successfully!')

df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_sh.csv')

print('Number of rows   :', df.shape[0])  # how many records
print('Number of columns:', df.shape[1])  # how many fields

All libraries imported successfully!
Number of rows   : 1424406
Number of columns: 19


In [2]:
df.isnull().sum()

order_id                  0
region_id                 0
city                      0
courier_id                0
accept_time               0
time_window_start         0
time_window_end           0
lng                       0
lat                       0
aoi_id                    0
aoi_type                  0
pickup_time               0
pickup_gps_time      441333
pickup_gps_lng       441333
pickup_gps_lat       441333
accept_gps_time      654425
accept_gps_lng       654425
accept_gps_lat       654425
ds                        0
dtype: int64

In [3]:
gps_cols = ['pickup_gps_lng', 'pickup_gps_lat',
            'accept_gps_lng',  'accept_gps_lat']
df[gps_cols] = df[gps_cols].fillna(0.0)

print(df[gps_cols].isnull().sum())

pickup_gps_lng    0
pickup_gps_lat    0
accept_gps_lng    0
accept_gps_lat    0
dtype: int64


In [4]:
# Step 1 — Reload the original file fresh (resets everything)
df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_sh.csv')

# Step 2 — Convert datetime columns (format is MM-DD HH:MM:SS, add year manually)
time_cols = ['accept_time', 'time_window_start', 'time_window_end', 'pickup_time']

for col in time_cols:
    df[col] = pd.to_datetime('2023-' + df[col],
                              format='%Y-%m-%d %H:%M:%S',
                              errors='coerce')

# Step 3 — Verify it worked
print(df[time_cols].dtypes)
print()
print(df['accept_time'].head())

accept_time          datetime64[ns]
time_window_start    datetime64[ns]
time_window_end      datetime64[ns]
pickup_time          datetime64[ns]
dtype: object

0   2023-07-08 08:13:00
1   2023-07-21 08:14:00
2   2023-07-12 07:40:00
3   2023-07-09 15:38:00
4   2023-07-07 07:25:00
Name: accept_time, dtype: datetime64[ns]


In [5]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [6]:
# Example: How many minutes from accept to pickup?
df['accept_to_pickup_min'] = (
    (df['pickup_time'] - df['accept_time']).dt.total_seconds() / 60
)
print(df['accept_to_pickup_min'].describe().round(1))
# This shows average, min, max pickup time in minutes

count    1424406.0
mean         229.7
std          442.8
min            0.0
25%           70.0
50%          124.0
75%          207.0
max        54859.0
Name: accept_to_pickup_min, dtype: float64


In [7]:
col = 'accept_to_pickup_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 158,158
Valid range: -136 to 412 minutes


In [8]:
df = df[df['accept_to_pickup_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

Rows after removing negatives: 1424406


In [9]:
df['accept_to_pickup_min'] = df['accept_to_pickup_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_pickup_min'].max())

Max duration after clipping: 412.5


In [10]:
df['accept_hour']  = df['accept_time'].dt.hour    # 0 to 23
df['accept_day']   = df['accept_time'].dt.day     # 1 to 31
df['accept_month'] = df['accept_time'].dt.month   # 1 to 12
print(df[['accept_time', 'accept_hour', 'accept_day', 'accept_month']].head(3))

          accept_time  accept_hour  accept_day  accept_month
0 2023-07-08 08:13:00            8           8             7
1 2023-07-21 08:14:00            8          21             7
2 2023-07-12 07:40:00            7          12             7


In [11]:
def time_of_day(hour):
    if   0 <= hour < 6:   return 'Night'
    elif 6 <= hour < 12:  return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                  return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())
print(df['time_of_day'].value_counts().sum())

time_of_day
Morning      1070242
Afternoon     347778
Evening         5934
Night            452
Name: count, dtype: int64
1424406


In [12]:
df['window_duration_min'] = (
    (df['time_window_end'] - df['time_window_start']).dt.total_seconds() / 60
)
print(df['window_duration_min'].value_counts().head())


window_duration_min
120.0     1400730
60.0         9360
899.0        6108
1841.0         27
1833.0         26
Name: count, dtype: int64


In [13]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min',
            'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month',
            'time_of_day', 'window_duration_min']
print(new_cols)

Final dataset shape: (1424406, 25)

Missing values left:
pickup_gps_time    441333
pickup_gps_lng     441333
pickup_gps_lat     441333
accept_gps_time    654425
accept_gps_lng     654425
accept_gps_lat     654425
dtype: int64

New columns created:
['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min', 'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'window_duration_min']


In [14]:
# Check all text columns and their unique values
text_cols = df.select_dtypes(include='object').columns.tolist()
print('Text columns:', text_cols)
print()

for col in text_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()
    

Text columns: ['city', 'pickup_gps_time', 'accept_gps_time', 'time_of_day']

--- city ---
city
Shanghai    1424406
Name: count, dtype: int64

--- pickup_gps_time ---
pickup_gps_time
07-01 09:44:00    36
07-05 10:34:00    36
06-23 10:20:00    36
07-16 09:12:00    35
07-18 10:04:00    34
                  ..
07-19 07:39:00     1
07-20 07:35:00     1
08-16 07:52:00     1
07-19 07:51:00     1
07-10 18:42:00     1
Name: count, Length: 103595, dtype: int64

--- accept_gps_time ---
accept_gps_time
06-19 08:35:00    123
06-21 08:45:00    109
06-21 08:46:00    103
06-25 08:36:00    100
06-21 08:36:00     97
                 ... 
06-14 15:20:00      1
09-07 15:29:00      1
06-18 07:02:00      1
10-26 06:42:00      1
08-18 15:06:00      1
Name: count, Length: 87635, dtype: int64

--- time_of_day ---
time_of_day
Morning      1070242
Afternoon     347778
Evening         5934
Night            452
Name: count, dtype: int64



In [15]:
for col in text_cols:
    df[col] = df[col].str.strip()        # remove spaces before/after
    df[col] = df[col].str.title()        # Capitalize First Letter
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)  # remove double spaces

print('Text columns cleaned!')
print(df['city'].value_counts())

Text columns cleaned!
city
Shanghai    1424406
Name: count, dtype: int64


In [16]:
# Cell 12b — Save cleaned data to a new CSV file
df.to_csv('pickup_sh_cleaned.csv', index=False)
print('File saved as: pickup_sh_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: pickup_sh_cleaned.csv
Total rows saved: 1,424,406


In [19]:
df.head()

,order_id,region_id,city,courier_id,accept_time,time_window_start,time_window_end,lng,lat,aoi_id,aoi_type,pickup_time,pickup_gps_time,pickup_gps_lng,pickup_gps_lat,accept_gps_time,accept_gps_lng,accept_gps_lat,ds,accept_to_pickup_min,accept_hour,accept_day,accept_month,time_of_day,window_duration_min
0,2349637,0,Shanghai,1448,2023-07-08 08:13:00,2023-07-08 09:00:00,2023-07-08 11:00:00,121.52223,30.90747,46,14,2023-07-08 10:10:00,07-08 10:10:00,121.52208,30.90836,NaN,NaN,NaN,708,117.0,8,8,7,Morning,120.0
1,4867696,0,Shanghai,1448,2023-07-21 08:14:00,2023-07-21 09:00:00,2023-07-21 11:00:00,121.52223,30.90742,46,14,2023-07-21 10:10:00,07-21 10:10:00,121.52689,30.91897,07-21 08:14:00,121.50334,30.90424,721,116.0,8,21,7,Morning,120.0
2,5691514,0,Shanghai,1448,2023-07-12 07:40:00,2023-07-12 17:00:00,2023-07-12 19:00:00,121.52229,30.90731,46,14,2023-07-12 17:22:00,07-12 17:22:00,121.52612,30.91764,07-12 07:37:00,121.49739,30.90695,712,412.5,7,12,7,Morning,120.0
3,1443776,0,Shanghai,1448,2023-07-09 15:38:00,2023-07-09 17:00:00,2023-07-09 19:00:00,121.52234,30.90749,46,14,2023-07-09 15:54:00,07-09 15:54:00,121.52316,30.90876,07-09 15:38:00,121.51814,30.90612,709,16.0,15,9,7,Afternoon,120.0
4,1806717,0,Shanghai,1448,2023-07-07 07:25:00,2023-07-07 09:00:00,2023-07-07 11:00:00,121.52230,30.90747,46,14,2023-07-07 09:53:00,07-07 09:53:00,121.51871,30.90687,07-07 07:25:00,121.49736,30.90752,707,148.0,7,7,7,Morning,120.0
